In [ ]:
import pandas as pd

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
from google.colab import drive
drive.mount

<function google.colab.drive.mount(mountpoint, force_remount=False, timeout_ms=120000, readonly=False)>

In [ ]:
filepath="/content/drive/MyDrive/Superstore.csv"
df =  pd.read_csv(filepath,encoding="latin1")

In [ ]:
pd.set_option('display.max_columns', None)
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2013-152156,09-11-2013,12-11-2013,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2013-152156,09-11-2013,12-11-2013,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2013-138688,13-06-2013,17-06-2013,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2012-108966,11-10-2012,18-10-2012,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2012-108966,11-10-2012,18-10-2012,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [ ]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

1. From the Superstore dataset, compute the top 5 states by total profit along with the number of unique customers in each of those states.

In [ ]:
top_5 = df.groupby('State').agg(Total_Profit=('Profit','sum'), Customers=('Customer Name','nunique')).sort_values(by='Total_Profit',ascending=False)
print(top_5.head())

            Total_Profit  Customers
State                              
California    76381.3871        577
New York      74038.5486        415
Washington    33402.6517        224
Michigan      24463.1876        106
Virginia      18597.9504        107


2. Identify the customer with the highest number of unique orders and display their total sales grouped by category.

In [ ]:
cust = df.groupby('Customer Name')["Order ID"].nunique().sort_values(ascending=False)
cust_max=cust.head(1)
print(cust_max)
df_cust=df[df['Customer Name'].isin(cust_max.index)]
# print(df_cust)  #returns dataframe
df_cust.groupby('Category')['Sales'].sum()


Customer Name
Emily Phan    17
Name: Order ID, dtype: int64


,Sales
Category,
Furniture,2860.0848
Office Supplies,1258.1180
Technology,1359.8580


In [ ]:
sale_Sum=df_cust.groupby('Category')['Sales'].sum()
print(sale_Sum)

Category
Furniture          2860.0848
Office Supplies    1258.1180
Technology         1359.8580
Name: Sales, dtype: float64


 3. Create a pivot table of average discount per ship mode, then rank ship modes by total profit. Compare whether higher average discounts align with higher profitability.





In [ ]:
pivot =  df.pivot_table(index='Ship Mode',values='Discount',aggfunc='mean')
print(pivot)

                Discount
Ship Mode               
First Class     0.164610
Same Day        0.152394
Second Class    0.138895
Standard Class  0.160023


In [ ]:
df.groupby("Ship Mode")['Discount'].mean()

,Discount
Ship Mode,
First Class,0.164610
Same Day,0.152394
Second Class,0.138895
Standard Class,0.160023


4. Divide Sales into quartiles (Q1–Q4) and within each quartile, find the most frequent sub-category by order count. Present the results for all quartiles side by side.

In [ ]:
df['Sales_Q'] = pd.qcut(df['Sales'],4,labels=("Q1","Q2","Q3","Q4"))
df.groupby('Sales_Q')['Sub-Category'].value_counts()

/tmp/ipython-input-345371279.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('Sales_Q')['Sub-Category'].value_counts()


Sales_Q  Sub-Category
Q1       Binders         711
         Paper           437
         Art             434
         Furnishings     212
         Labels          202
                        ... 
Q4       Supplies         26
         Envelopes        17
         Art              12
         Labels            8
         Fasteners         0
Name: count, Length: 68, dtype: int64

5. Calculate the Interquartile Range (IQR) of Profit and use it to detect outlier profits. List the categories that contain the largest proportion of such outliers.

In [ ]:
q25 =  df['Profit'].quantile(0.25)
q75 =  df['Profit'].quantile(0.75)
iqr =  q75 - q25
print(iqr)
lower_bound = q25 - 1.5*iqr
upper_bound = q75 + 1.5*iqr
df_outlier =  df[(df['Profit'] <= lower_bound) | (df['Profit'] >= upper_bound)]
print(df_outlier)

27.63525
      Row ID        Order ID  Order Date   Ship Date       Ship Mode  \
1          2  CA-2013-152156  09-11-2013  12-11-2013    Second Class   
3          4  US-2012-108966  11-10-2012  18-10-2012  Standard Class   
7          8  CA-2011-115812  09-06-2011  14-06-2011  Standard Class   
10        11  CA-2011-115812  09-06-2011  14-06-2011  Standard Class   
13        14  CA-2013-161389  06-12-2013  11-12-2013  Standard Class   
...      ...             ...         ...         ...             ...   
9957    9958  US-2011-143287  11-11-2011  17-11-2011  Standard Class   
9962    9963  CA-2012-168088  19-03-2012  22-03-2012     First Class   
9968    9969  CA-2014-153871  12-12-2014  18-12-2014  Standard Class   
9979    9980  US-2013-103674  07-12-2013  11-12-2013  Standard Class   
9993    9994  CA-2014-119914  05-05-2014  10-05-2014    Second Class   

     Customer ID     Customer Name      Segment        Country  \
1       CG-12520       Claire Gute     Consumer  United Stat

In [ ]:
# value_counts()
category = df_outlier.groupby('Category').size()
print(category)

Category
Furniture          627
Office Supplies    685
Technology         569
dtype: int64


6. Filter the dataset for California and Texas. Compare the mean profit per order for both states, and identify which state shows higher standard deviation of profit.

In [ ]:
df_cali=df[df['State'].isin(['California','Texas'])]
print(df_cali)


      Row ID        Order ID  Order Date   Ship Date       Ship Mode  \
2          3  CA-2013-138688  13-06-2013  17-06-2013    Second Class   
5          6  CA-2011-115812  09-06-2011  14-06-2011  Standard Class   
6          7  CA-2011-115812  09-06-2011  14-06-2011  Standard Class   
7          8  CA-2011-115812  09-06-2011  14-06-2011  Standard Class   
8          9  CA-2011-115812  09-06-2011  14-06-2011  Standard Class   
...      ...             ...         ...         ...             ...   
9986    9987  CA-2013-125794  30-09-2013  04-10-2013  Standard Class   
9990    9991  CA-2014-121258  27-02-2014  04-03-2014  Standard Class   
9991    9992  CA-2014-121258  27-02-2014  04-03-2014  Standard Class   
9992    9993  CA-2014-121258  27-02-2014  04-03-2014  Standard Class   
9993    9994  CA-2014-119914  05-05-2014  10-05-2014    Second Class   

     Customer ID    Customer Name    Segment        Country         City  \
2       DV-13045  Darrin Van Huff  Corporate  United States

In [ ]:
mean = df_cali.groupby('State')['Profit'].mean().sort_values(ascending=False)
std = df_cali.groupby('State')['Profit'].std().sort_values(ascending=False)
print(mean)
print(std.head(2))

State
California    38.171608
Texas        -26.121174
Name: Profit, dtype: float64
State
Texas         189.022781
California     97.691593
Name: Profit, dtype: float64


7. Add a new column labeling each row as Profit or Loss. For each region, compute the ratio of profitable to loss-making orders and determine the region with the healthiest ratio.

In [ ]:
# df["Profit_Loss"] = pd.apply(lambda x: "Positive" if df['Profit'] > 0 else "Negative")

df['Profit_Loss'] = df['Profit'].apply(lambda x: 'Profit' if x > 0 else 'Loss')
print(df)

      Row ID        Order ID  Order Date   Ship Date       Ship Mode  \
0          1  CA-2013-152156  09-11-2013  12-11-2013    Second Class   
1          2  CA-2013-152156  09-11-2013  12-11-2013    Second Class   
2          3  CA-2013-138688  13-06-2013  17-06-2013    Second Class   
3          4  US-2012-108966  11-10-2012  18-10-2012  Standard Class   
4          5  US-2012-108966  11-10-2012  18-10-2012  Standard Class   
...      ...             ...         ...         ...             ...   
9989    9990  CA-2011-110422  22-01-2011  24-01-2011    Second Class   
9990    9991  CA-2014-121258  27-02-2014  04-03-2014  Standard Class   
9991    9992  CA-2014-121258  27-02-2014  04-03-2014  Standard Class   
9992    9993  CA-2014-121258  27-02-2014  04-03-2014  Standard Class   
9993    9994  CA-2014-119914  05-05-2014  10-05-2014    Second Class   

     Customer ID     Customer Name    Segment        Country             City  \
0       CG-12520       Claire Gute   Consumer  United 

8. Generate the list of top 10 most profitable products. Then, compare it with the sub-categories that have the lowest average profit. Report whether any product from these low-profit sub-categories still appears in the overall top 10 list.  

In [ ]:
list_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)
print(list_profit)

sub_cat = df.groupby('Sub-Category')['Profit'].mean().sort_values(ascending=True).head(10)
print(sub_cat)

#Comparision
print(list_profit.index.isin(sub_cat.index))

Product Name
Canon imageCLASS 2200 Advanced Copier                                          25199.9280
Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind     7753.0390
Hewlett Packard LaserJet 3310 Copier                                            6983.8836
Canon PC1060 Personal Laser Copier                                              4570.9347
HP Designjet T520 Inkjet Large Format Printer - 24" Color                       4094.9766
Ativa V4110MDD Micro-Cut Shredder                                               3772.9461
3D Systems Cube Printer, 2nd Generation, Magenta                                3717.9714
Plantronics Savi W720 Multi-Device Wireless Headset System                      3696.2820
Ibico EPK-21 Electric Binding System                                            3345.2823
Zebra ZM400 Thermal Label Printer                                               3343.5360
Name: Profit, dtype: float64
Sub-Category
Tables        -55.565771
Bookcases     -15.23